# Prompting

Prompting is an essential component when working with LLMs, and Agents SDK naturally has it's own way of handling various the components of prompts. In this chapter, we'll look at _how_ to use static and dynamic prompting, how to correctly use system, user, assistant, and tool prompts. Then, we'll see how these come together to create conversational agents.

Now we create an agent, in Agents SDK we do this via the `Agent` class. When initializing an `Agent` we include a few parameters:

- `name` is naturally the agent's name. This is referenced by the agent (for example if you ask it's name) but otherwise is more of an identifier for us.
- `instructions` is the system prompt which guides the behavior of the agent.
- `model` is the model to be used, we're using `gpt-4.1-mini` as it's a strong yet fast and cheap model.

In [1]:
import os
from agents import Agent, Runner
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
from dotenv import load_dotenv

load_dotenv()


True

In [2]:
# Setting up the NVIDIA NIM client
client = AsyncOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)

In [3]:
from agents import Agent 

agent = Agent(
    name="My Agent",
    model=OpenAIChatCompletionsModel(
        model="qwen/qwen3.5-122b-a10b",
        openai_client=client
    ),
    instructions="You are a helpful assistant."
)

Now we want to run the agent, Agents SDK provides a `Runner` object that will allow us to run the agent

However we need to use the `await` keyword to run the agent, this is because the `Runner` object is an asynchronous object

In the `run` method we have the following parameters:

- `starting_agent` defines which agent our workflow begins with. In this case, we only have a single agent workflow but in more complex scenarios we may find ourselves using many agents in a single workflow run, and in that scenario we would also have a specific `starting_agent` that may handover to our other agents.
- `input`: The input to pass to the agent, typically our user query.

We'll ask our agent to write us a haiku. A haiku is a traditional form of Japanese poetry, which follows a 5-7-5 syllable pattern. Typically, a haiku should invoke some sense of a window into a broader world, such as making you think of the rain as it splashes into a pond, or the wind as it flows through the trees in a forest — traditionally haikus also tend to focus on the natural world.

In [4]:
from agents import Runner

result = await Runner.run(
    starting_agent=agent, 
    input="Write me a haiku about the ocean."
)

print(result.final_output)

Deep blue waves roll in,
Salt spray kisses the warm sand,
Endless tides breathe soft.


OPENAI_API_KEY is not set, skipping trace export


## Dynamic Instructions

Now we can take this a step further and provide a dynamic system prompt to the agent. With dynamic system prompts / `instructions` we can modify what is passed to the agent based on some dynamic parameter which is filled at query time.

First, we create a function that will construct our dynamic prompt. This function will simply provide the current time to the agent, and then ask the agent to change it's behavior based on the time that is provided.

In [5]:
from datetime import datetime
from agents import RunContextWrapper

def time_based_instructions(
    context: RunContextWrapper, 
    agent: Agent
) -> str:
    time = datetime.now().strftime("%H:%M")
    return (
        f"The current time is {time}. If it is the afternoon, speak like a pirate, otherwise "
        "do not."
    )

Next we need to redefine our agent with this new dynamic system prompt. To do this we pass the `time_based_instructions` function to our `instructions` parameter. Note that we pass the function itself to `instructions`, which is then called at query time _not_ when we initialize the agent.

In [6]:
agent = Agent(
    name="Time Agent",
    instructions=time_based_instructions,  # note we're passing the function itself
    model=OpenAIChatCompletionsModel(
        model="qwen/qwen3.5-122b-a10b",
        openai_client=client
    ),
)

Then using the Runner object we can test our dynamic instructions

In [7]:
result = await Runner.run(
    starting_agent=agent,
    input="Hello, what time is it?"
)
result.final_output

OPENAI_API_KEY is not set, skipping trace export


'It is 01:47. Since that is in the early morning (not the afternoon), I will speak normally:\n\nIt is 1:47 AM.'

OPENAI_API_KEY is not set, skipping trace export


For this function to be `dynamic` you can see that when asking the time, the agent will return a different response based on the time of day without having to re-initialize the agent. We can ask for another haiku too:

In [8]:
result = await Runner.run(
    starting_agent=agent,
    input="Write me a haiku"
)
print(result.final_output)

Soft moonlight descends,
Silent stars above the night,
Dreams begin to wake.


OPENAI_API_KEY is not set, skipping trace export


# Message Types

We're going to using _five_ primary message types from OpenAI, those are:

* `user` is almost always just the input query from a user. Occasionally we might modify this in some way but this isn't particularly common — so we can assume this is the direct input query from a user.
* `developer` is used to provide instructions directly to the LLM. Typically these are where we would put behavioral instructions, or rules and parameters for how we'd like a conversation with the LLM to be executed. In the past these were called `system` messages.
* `assistant` is the direct response from an LLM to a user.
* `function_call` is the response from an LLM in the scenario where the LLM has decided it would like to use a tool / function call. Many frameworks will structure this as an `assistant` message with an additional tool call field — but with OpenAI and Agents SDK these are their own message type.
* `function_call_output` is the output from our executed tool / function. It is typically constructed within our codebase as OpenAI is _not_ executing our code for us.

In a typical conversation with an agent that has tools, we might see the following:

![Event list of interactions with an agent that has tools](../assets/prompting-events-dark.png)

It's worth clarifying that _technically_ we have just listed _three_ message types. The `user`, `developer`, and `assistant` messages are all of the same message _type_, which is `type="message"`. These three messages are distinguished as different having _roles_, meaning they are all of `type="message"` but are of different roles, ie `role="user"`, `role="developer"`, or `role="assistant"`.

Now, Agents SDK will abstract away the majority of these message types for us. In fact, during typical use of the framework we'll typically define an initial `developer` message via the `instructions` field of our `Agent` object, and we'll define `user` messages via the `input` field of our `Runner.run` method.

We would not necessarily need to know these other message types to use Agents SDK. Fortunately, there are easy-to-use methods such as the `to_input_list()` method that will take the outputs we receive from Agents SDK and format them into the format we need for feeding them back into the `input` parameter.

However, by not understanding these message types and how they are used by Agents SDK we would (1) have less understanding of how the system we're building truly works, which can be important particularly with prompting and designing a good agent workflow. And (2) when pulling in messages from other places, such as our own databases or simply via our own code logic, we do need to construct our own chat history using these message types.

So, although not 100% necessary, we think it's still pretty important to understand message types _well_ and practically essential for most production use-cases.

## User Messages

Beginning with our `user` message. The `user` messages are automatically defined when we call our runner via the `input` parameter:

In [9]:
result = await Runner.run(
    starting_agent=agent,
    input="Write me a haiku"  # this creates a user message
)
print(result.final_output)

Soft moon watches high,
Silent stars begin to fade,
Night whispers goodbye.


OPENAI_API_KEY is not set, skipping trace export


Alternatively, if we'd like to use typing we can import the `Message` object directly from the `openai` library (which is used under-the-hood by Agents SDK).

In [10]:
from openai.types.responses.response_input_item_param import Message

user_message = Message(
    role="user",
    content="write me a haiku",
    type="message",
    status="completed"
)

In [11]:
result = await Runner.run(
    starting_agent=agent,
    input=[user_message]
)
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


Soft moon lights the night,
Silent stars above the dark,
Time to close your eyes.


OPENAI_API_KEY is not set, skipping trace export


In most cases, we can simplify all of this and directly define user messages using the dictionary format:

In [12]:
user_message = {"role": "user", "content": "write me a haiku"}

# Developer Messages

The `developer` message defines how the agent should behave. This message was previously called the `system` message but for models **o1** and newer the `developer` message should be used in it's place. The initial `developer` message is automatically added to our agents when we define the `Agent` object and it is defined via the `instructions` parameter:

In [13]:
agent = Agent(
    name="Agent",
    instructions="Talk like a pirate",  # here is our initial system/developer prompt
 model=OpenAIChatCompletionsModel(
        model="qwen/qwen3.5-122b-a10b",
        openai_client=client
    ),
)

We can also define a developer message directly by setting `role="developer"` in a dictionary like so:

In [14]:
developer_msg = {"role": "developer", "content": "Talk like a pirate"}

However, it's worth noting that the instructions / first system prompt cannot be set other than via the `instructions` parameter. Instead we would likely use the system message to add additional instructions within our chat history, which might look something like this:

In [ ]:
result = await Runner.run(
    starting_agent=agent,
    input=[
        {"role": "user", "content": "write me a haiku"},
        {
            "role": "developer", # this model do not support this model
            "content": "Don't speak like a pirate and instead use obvious British slang"
        }
    ]
)
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


BadRequestError: Error code: 400 - {'error': {'message': 'System message must be at the beginning.', 'type': 'BadRequestError', 'param': None, 'code': 400}}